# Tutorial 05 — Invariantes de Ciclo de Vida, Casos Negativos (`N-<CÓDIGO>`) y Degradación Operativa

El catálogo cerrado **F5** (`internal/core/codes/catalog.tsv`) define 64 códigos normativos para cada condición de error en el Hub y en el cliente.
En este notebook probaremos mediante peticiones HTTP:
1. **Invariantes de la tabla de 9 filas de aprobación (`REQ-HUB-005`)**: `DATES_GRACE_ON_COMPATIBLE`, `DATES_EMERGENCY_ON_COMPATIBLE`, `DATES_ZERO_GRACE_WITHOUT_EMERGENCY`, `DATES_JUSTIFICATION_REQUIRED`, `DATES_EMERGENCY_WITH_GRACE`.
2. **Invariantes de revocación (`REQ-HUB-014`)**: `DATES_REVOKE_LEAVES_NO_HEAD` (`REV-4`), `DATES_JUSTIFICATION_REQUIRED` (`REV-2`), `DATES_REVOCATION_TOO_EARLY` (`REV-3`) y `DATES_ALREADY_REVOKED` (`REV-1`).
3. **Máquina de estados del cliente (`Active` -> `Degraded` -> `FailClosed`, `REQ-CLI-008`, `REQ-CLI-018`, `REQ-SYS-008`)** cuando el Hub deja de responder y el listado caduca.

In [1]:
import json
from lab import AlexandriaLab, http_request, show_state_machine

# Use compiler="poc" to build concise JSON drafts while testing all date combinations
lab = AlexandriaLab(compiler="poc", initial_clock="2026-09-01T10:00:00Z")
lab.start_all()

def make_poc_draft(version: str, actions: list[str]) -> dict:
    return {
        "space": "https://example.com/spaces/demo",
        "version": version,
        "slots": [{"slot": "odrl", "obligation": "mandatory", "profiles": [f"https://example.com/profiles/odrl/{version}"]}],
        "files": [
            {
                "path": "schemas/policy.schema.json",
                "media_type": "application/schema+json",
                "role": "json-schema",
                "slot": "odrl",
                "applies_to": ["policy_intake"],
                "content": {
                    "type": "object",
                    "properties": {"action": {"enum": actions}},
                    "required": ["action"],
                },
            }
        ],
    }

def propose_draft(doc: dict) -> str:
    _, _, d = http_request("POST", f"{lab.hub_url}/v1/drafts", token="editor-token", body=doc, verbose=False)
    _, _, c = http_request("POST", f"{lab.hub_url}/v1/drafts/{d['id']}:propose", token="editor-token", verbose=False)
    return c["id"]

# Publish initial 1.0.0 release and advance the clock to September 5
c1 = propose_draft(make_poc_draft("1.0.0", ["odrl:use", "ex:access"]))
http_request("POST", f"{lab.hub_url}/v1/candidates/{c1}:approve", token="releaser-token", body={}, verbose=False)
lab.set_clock("2026-09-05T12:00:00Z")
lab.wait_synced("2026-09-05T12:00:00Z")
print("✔ Versión 1.0.0 publicada y sincronizada en el cliente")

✔ Versión 1.0.0 publicada y sincronizada en el cliente


✔ alexandria-server listening at http://127.0.0.1:60549 (DID: did:web:127.0.0.1%3A60549)

✔ alexandria-client active | dsp-ingress: http://127.0.0.1:60551 | dsp-egress: http://127.0.0.1:60553 | admin: http://127.0.0.1:60554

🕒 Clock updated → 2026-09-05T12:00:00Z

## 1. Invariantes de Aprobación en Versiones Compatibles (`MINOR` / `PATCH`)

Si proponemos una versión compatible `1.1.0` (que añade `odrl:read` sin quitar `ex:access`):
- No admite `grace_window` explícita -> **`DATES_GRACE_ON_COMPATIBLE`**.
- No admite `emergency: true` -> **`DATES_EMERGENCY_ON_COMPATIBLE`**.

In [2]:
cand_110 = propose_draft(make_poc_draft("1.1.0", ["odrl:use", "ex:access", "odrl:read"]))

print("1) Aprobar versión MINOR 1.1.0 con grace_window explícita -> DATES_GRACE_ON_COMPATIBLE:")
st1, _, b1 = http_request(
    "POST",
    f"{lab.hub_url}/v1/candidates/{cand_110}:approve",
    token="releaser-token",
    body={"grace_window": "P10D"},
)
assert b1["code"] == "DATES_GRACE_ON_COMPATIBLE"

print("2) Aprobar versión MINOR 1.1.0 con emergency: true -> DATES_EMERGENCY_ON_COMPATIBLE:")
st2, _, b2 = http_request(
    "POST",
    f"{lab.hub_url}/v1/candidates/{cand_110}:approve",
    token="releaser-token",
    body={"emergency": True, "justification": "test"},
)
assert b2["code"] == "DATES_EMERGENCY_ON_COMPATIBLE"

1) Aprobar versión MINOR 1.1.0 con grace_window explícita -> DATES_GRACE_ON_COMPATIBLE:
2) Aprobar versión MINOR 1.1.0 con emergency: true -> DATES_EMERGENCY_ON_COMPATIBLE:


connection,close
content-length,225
content-type,application/problem+json
date,"Sat, 05 Sep 2026 12:00:00 GMT"


connection,close
content-length,234
content-type,application/problem+json
date,"Sat, 05 Sep 2026 12:00:00 GMT"


## 2. Invariantes de Aprobación en Versiones que Rompen (`MAJOR` `2.0.0`)

En una versión `MAJOR` (`2.0.0`):
- `grace_window: "P0D"` sin `emergency: true` -> **`DATES_ZERO_GRACE_WITHOUT_EMERGENCY`**.
- `emergency: true` sin `justification` -> **`DATES_JUSTIFICATION_REQUIRED`**.
- `emergency: true` combinada con `grace_window: "P10D"` -> **`DATES_EMERGENCY_WITH_GRACE`**.

In [3]:
cand_200 = propose_draft(make_poc_draft("2.0.0", ["odrl:use", "odrl:read"]))

for desc, payload, expected_code in [
    ("grace_window P0D sin emergency", {"grace_window": "P0D"}, "DATES_ZERO_GRACE_WITHOUT_EMERGENCY"),
    ("emergency sin justification", {"emergency": True}, "DATES_JUSTIFICATION_REQUIRED"),
    ("emergency con grace_window > 0", {"emergency": True, "grace_window": "P10D", "justification": "urgente"}, "DATES_EMERGENCY_WITH_GRACE"),
]:
    print(f"--- Caso: {desc} (esperado {expected_code}) ---")
    _, _, err_body = http_request("POST", f"{lab.hub_url}/v1/candidates/{cand_200}:approve", token="releaser-token", body=payload)
    assert err_body["code"] == expected_code

--- Caso: grace_window P0D sin emergency (esperado DATES_ZERO_GRACE_WITHOUT_EMERGENCY) ---
--- Caso: emergency sin justification (esperado DATES_JUSTIFICATION_REQUIRED) ---
--- Caso: emergency con grace_window > 0 (esperado DATES_EMERGENCY_WITH_GRACE) ---


connection,close
content-length,241
content-type,application/problem+json
date,"Sat, 05 Sep 2026 12:00:00 GMT"


connection,close
content-length,230
content-type,application/problem+json
date,"Sat, 05 Sep 2026 12:00:00 GMT"


connection,close
content-length,226
content-type,application/problem+json
date,"Sat, 05 Sep 2026 12:00:00 GMT"


## 3. Invariantes de Revocación (`REV-1` a `REV-4`, `REQ-HUB-014`)

1. **`REV-4` (`DATES_REVOKE_LEAVES_NO_HEAD`):** Intentar revocar `1.0.0` cuando es la única versión vigente.
2. Tras aprobar `2.0.0` y avanzar el reloj para que `2.0.0` sea cabeza vigente:
   - **`REV-2` (`DATES_JUSTIFICATION_REQUIRED`):** Revocar `1.0.0` sin justificación.
   - **`REV-3` (`DATES_REVOCATION_TOO_EARLY`):** Revocar `1.0.0` con `revoked_from` anterior a `now + propagation_window` (`P2D`).
   - **`REV-1` (`DATES_ALREADY_REVOKED`):** Revocar `1.0.0` una segunda vez.

In [4]:
print("1) Intentar revocar 1.0.0 siendo la única versión vigente (esperado DATES_REVOKE_LEAVES_NO_HEAD):")
_, _, b_rev4 = http_request(
    "POST",
    f"{lab.hub_url}/v1/versions/1.0.0:revoke",
    token="releaser-token",
    body={"justification": "intento invalido"},
)
assert b_rev4["code"] == "DATES_REVOKE_LEAVES_NO_HEAD"

# Approve 2.0.0 and advance the clock to 2026-10-10 so 2.0.0 is active
http_request("POST", f"{lab.hub_url}/v1/candidates/{cand_200}:approve", token="releaser-token", body={}, verbose=False)
lab.set_clock("2026-10-10T12:00:00Z")

print("2) Revocar 1.0.0 sin justificación (esperado DATES_JUSTIFICATION_REQUIRED):")
_, _, b_rev2 = http_request("POST", f"{lab.hub_url}/v1/versions/1.0.0:revoke", token="releaser-token", body={})
assert b_rev2["code"] == "DATES_JUSTIFICATION_REQUIRED"

print("3) Revocar 1.0.0 con fecha demasiado temprana < now + P2D (esperado DATES_REVOCATION_TOO_EARLY):")
_, _, b_rev3 = http_request(
    "POST",
    f"{lab.hub_url}/v1/versions/1.0.0:revoke",
    token="releaser-token",
    body={"justification": "motivo", "revoked_from": "2026-10-11T00:00:00Z"},
)
assert b_rev3["code"] == "DATES_REVOCATION_TOO_EARLY"

print("4) Revocar 1.0.0 correctamente y volver a revocar (esperado DATES_ALREADY_REVOKED):")
st_ok, _, _ = http_request(
    "POST",
    f"{lab.hub_url}/v1/versions/1.0.0:revoke",
    token="releaser-token",
    body={"justification": "revocacion valida"},
    verbose=False,
)
assert st_ok == 200
_, _, b_rev1 = http_request(
    "POST",
    f"{lab.hub_url}/v1/versions/1.0.0:revoke",
    token="releaser-token",
    body={"justification": "segunda vez"},
)
assert b_rev1["code"] == "DATES_ALREADY_REVOKED"

1) Intentar revocar 1.0.0 siendo la única versión vigente (esperado DATES_REVOKE_LEAVES_NO_HEAD):
2) Revocar 1.0.0 sin justificación (esperado DATES_JUSTIFICATION_REQUIRED):
3) Revocar 1.0.0 con fecha demasiado temprana < now + P2D (esperado DATES_REVOCATION_TOO_EARLY):
4) Revocar 1.0.0 correctamente y volver a revocar (esperado DATES_ALREADY_REVOKED):


connection,close
content-length,234
content-type,application/problem+json
date,"Sat, 05 Sep 2026 12:00:00 GMT"


🕒 Clock updated → 2026-10-10T12:00:00Z

connection,close
content-length,222
content-type,application/problem+json
date,"Sat, 10 Oct 2026 12:00:00 GMT"


connection,close
content-length,245
content-type,application/problem+json
date,"Sat, 10 Oct 2026 12:00:00 GMT"


connection,close
content-length,188
content-type,application/problem+json
date,"Sat, 10 Oct 2026 12:00:00 GMT"


## 4. Caída del Hub y Transición `Active -> Degraded -> FailClosed` (`REQ-SYS-008`, `REQ-CLI-018`)

¿Qué sucede en `alexandria-client` si `alexandria-server` cae o queda inalcanzable durante días?
1. Sincronizamos el cliente en `2026-10-10T12:00:00Z` (el listado caduca en `expires_at = 2026-10-11T12:00:00Z`, `TTL = 24h`, y `max_staleness = P7D`).
2. **Detenemos `alexandria-server`**.
3. Avanzamos el reloj a **`2026-10-12T00:00:00Z`** (`expires_at` vencido, pero dentro de `max_staleness`): el cliente entra en estado **`Degraded`**.
4. Avanzamos el reloj a **`2026-10-20T00:00:00Z`** (superado `expires_at + max_staleness`): el cliente entra en **`FailClosed`** y rechaza todas las rutas `enforce` con **`CLIENT_FAIL_CLOSED`** (`REQ-SYS-008`).

In [5]:
lab.wait_synced("2026-10-10T12:00:00Z")

# Terminate the alexandria-server process (second process started in lab.procs)
lab.procs[1].terminate()
lab.procs[1].wait(timeout=3)
print("🔌 alexandria-server detenido para simular caída de red")

# 1) Advance clock to 2026-10-12T00:00:00Z -> Degraded state
lab.set_clock("2026-10-12T00:00:00Z")
import time
time.sleep(1.2)
_, _, st_deg = http_request("GET", f"{lab.admin_url}/status")
assert st_deg["state"] == "Degraded"
show_state_machine(st_deg["state"], subtitle="expires_at vencido (dentro de max_staleness P7D)")

# 2) Advance clock to 2026-10-20T00:00:00Z -> FailClosed state (past TTL + max_staleness P7D)
lab.set_clock("2026-10-20T00:00:00Z")
time.sleep(1.2)
_, _, st_fc = http_request("GET", f"{lab.admin_url}/status")
assert st_fc["state"] == "FailClosed"
show_state_machine(st_fc["state"], subtitle="Superado expires_at + max_staleness -> CLIENT_FAIL_CLOSED")

print("3) Enviar petición DSP cuando el cliente está en fail_closed (esperado 400 CLIENT_FAIL_CLOSED):")
st_req, hdr_req, body_req = http_request(
    "POST",
    f"{lab.dsp_ingress_url}/negotiations/request",
    body={"@type": "ContractRequestMessage", "consumerPid": "urn:uuid:c1", "offer": {"action": "odrl:use"}},
)
assert st_req == 400
assert body_req["code"] == "CLIENT_FAIL_CLOSED"

lab.close()

🔌 alexandria-server detenido para simular caída de red
3) Enviar petición DSP cuando el cliente está en fail_closed (esperado 400 CLIENT_FAIL_CLOSED):


🕒 Clock updated → 2026-10-12T00:00:00Z

connection,close
content-length,613
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:34 GMT"


🛡️ Client operational state: Degraded (expires_at vencido (dentro de max_staleness P7D))

🕒 Clock updated → 2026-10-20T00:00:00Z

connection,close
content-length,615
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:35 GMT"


🛡️ Client operational state: FailClosed (Superado expires_at + max_staleness -> CLIENT_FAIL_CLOSED)

connection,close
content-length,185
content-type,application/json
date,"Mon, 09 Nov 2026 22:45:35 GMT"
vhub-verdict,"violation;gate=dsp_message;code=""CLIENT_FAIL_CLOSED"", violation;gate=policy_intake;code=""CLIENT_FAIL_CLOSED"""
